# Appendix C — Calibrating GMS Decision Thresholds

*The chapter says calibrate it. This appendix shows how.*

## The thresholds that matter

| Threshold | What it controls | Default |
|---|---|---|
| `theta_plausibility` | Plausibility-gate cut on `score_triple` | 1.5 |
| `tau_contra` | Contradiction cut on `tension_energy` | ~1.7 |
| `numeric_tolerance` | NUMERIC-claim tolerance vs ENM | 0.01 |
| `holonomy_threshold` | Multi-hop escalation threshold | 0.5 |
| `epsilon_phase` | Inequality slack on phase encoder | 0.05 rad |

We walk through `theta_plausibility` against a labeled cohort, using the DOE machinery from Chapter 11.

In [ ]:
import torch
from dataclasses import dataclass
from pathlib import Path
from knowlytix.knowledge.query import DocGMSConfig, GMSExpertStore
from proofloop.evaluation import balanced_design, coverage_report

def _find_store(name):
    for p in [Path.cwd(), *Path.cwd().parents]:
        for stem in ["beyond-prompt-and-pray", "GMS-Agents/beyond-prompt-and-pray"]:
            c = p / stem / "code" / "data" / name
            if c.exists():
                return c
    raise FileNotFoundError(f"{name} not found")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = DocGMSConfig(store_path=str(_find_store('gms_banking_store')))
store = GMSExpertStore(config, device=DEVICE)
store.load()


## A labeled cohort

Calibration needs labeled positives and negatives. The cohort below uses the workflow steps from the banking policy; positive examples are real workflow edges, negative examples are nonsense edges. Production cohorts are sampled from labeled production traces.

In [ ]:
@dataclass
class LabeledEdge:
    context: str
    relation: str
    tail: str
    label: str  # 'admit' or 'block'

cohort = [
    # Real workflow edges -> admit
    LabeledEdge('start',         'has_enables', 'classify',     'admit'),
    LabeledEdge('classify',      'has_enables', 'extract',      'admit'),
    LabeledEdge('extract',       'has_enables', 'search_policy','admit'),
    LabeledEdge('extract',       'has_enables', 'flag_regulatory','admit'),
    # Nonsense edges -> block
    LabeledEdge('classify',      'has_enables', 'wire_international','block'),
    LabeledEdge('classify',      'has_enables', 'stop_payment',  'block'),
    LabeledEdge('classify',      'has_enables', 'draft_response','block'),
    LabeledEdge('overdraft',     'has_enables', 'classify',     'block'),
]
n_admit = sum(1 for c in cohort if c.label == 'admit')
n_block = sum(1 for c in cohort if c.label == 'block')
print(f'cohort: {len(cohort)} edges ({n_admit} admit, {n_block} block)')

## The sweep

For each candidate threshold value, score each cohort edge through `store.score_triple` and count false admits (block-labeled edges passing) and false rejects (admit-labeled edges failing). The threshold that minimizes both is the operating point.

In [ ]:
thetas = [0.5, 0.7, 0.9, 1.1, 1.3, 1.5, 1.7, 1.9]
rows = []
for theta in thetas:
    fa, fr, tp, tn = 0, 0, 0, 0
    for edge in cohort:
        s = store.score_triple(edge.context, edge.relation, edge.tail)
        if s is None:
            continue  # on_missing path; treat as no decision
        admitted = s <= theta
        if admitted and edge.label == 'block': fa += 1
        elif not admitted and edge.label == 'admit': fr += 1
        elif admitted and edge.label == 'admit': tp += 1
        elif not admitted and edge.label == 'block': tn += 1
    rows.append({'theta': theta, 'false_admits': fa, 'false_rejects': fr,
                 'accuracy': (tp + tn) / max(tp+tn+fa+fr, 1)})
for r in rows:
    print(f'  theta={r["theta"]:.2f}  FA={r["false_admits"]}  FR={r["false_rejects"]}  acc={r["accuracy"]:.2f}')

## Pick an operating point

The cost of an error is asymmetric:

- **False admit** — an implausible call goes through. Downstream gates may catch it.
- **False reject** — a legitimate call is denied. The agent has to replan or escalate.

In a regulated setting the cost of a false admit dominates; pick the lowest threshold that achieves zero false admits.

In [ ]:
best = max(rows, key=lambda r: (r['accuracy'], -r['false_admits'], -r['false_rejects']))
print(f'best operating point: theta = {best["theta"]:.2f}  acc={best["accuracy"]:.2f}  FA={best["false_admits"]}  FR={best["false_rejects"]}')

## A DOE design over multiple thresholds

When more than one threshold is in play, the balanced design from Chapter 11 covers the joint factor space without running the full grid.

In [ ]:
factors = {
    'theta_plausibility': [1.0, 1.2, 1.5, 1.8],
    'tau_contra':         [1.5, 1.7, 1.9],
    'numeric_tolerance':  [0.005, 0.01, 0.02],
    'holonomy_threshold': [0.3, 0.5, 0.7],
}
design = balanced_design(factors, num_cases=12, seed=42)
print('design:')
for row in design[:4]:
    print(f'  {row}')
print('...')
print('coverage:')
for fname, counts in coverage_report(design, factors).items():
    print(f'  {fname}: {counts}')

## When to recalibrate

Recalibrate when the store has been re-trained, the drift monitor reports PSI above its threshold, the gate's false-reject rate creeps up, or domain rules change.

## Anti-patterns flagged here

- Calibrating on a cohort produced by the same agent.
- One-shot calibration. The substrate drifts; bundles expire.
- Choosing thresholds by eye on a precision-recall curve. Pick on the cost-weighted frontier.

In [ ]:
# Self-check: the sweep produced a sensible best threshold and the design is balanced.
assert best['accuracy'] >= 0.5
covered = coverage_report(design, factors)
for fname, counts in covered.items():
    assert max(counts.values()) - min(counts.values()) <= 1, f'{fname} not balanced'
print('OK')